# Knowledge retrieval

## Dependencies Setup

In [1]:
!pip install pdfplumber
!pip install autocorrect
!pip install stanza
!pip install PyMuPDF
!pip install transformers==4.12.0
!pip install PyPDF2
!pip install transformers
!pip install textacy
!pip install rouge|
!pip install sentence-transformers
!pip install faiss-cpu --no-cache

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 51.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 67.5 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.8/622.8 kB 8.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for autocorrect: filename=autocorrect-2.6.1-py3-none-any.whl size=622363 sha256=2007e3a9e742467752ac437a9d53edb2b5d29496d3178b0f3ac84adba2e813af
  Stored in directory: /root/.cache/pip/wheels/b5/7b/6d/b76b29ce11ff8e2521c8c7dd0e5bfee4fb1789d76193124343
Successfully built autocorrect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 69.7 MB/s eta 0:00

In [2]:
!pip install rouge-score
!pip install pdfplumber
!pip install stanza
!pip install transformers
!pip install rouge

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=adb3f5ca31780b580f3c7c3e6ee1f499ddb9e639bd96c049e3b8a412b401dca7
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


## Library Imports

In [3]:
from rouge import Rouge
import  pdfplumber
import string
import re
import stanza
from transformers import pipeline
import pandas as pd
#Download the Stanza model for your desired language (e.g., English)
stanza.download('en')
nlp = stanza.Pipeline(lang='en', processors='tokenize,pos')
#fix_spelling = pipeline("text2text-generation",model="oliverguhr/spelling-correction-english-base")

/opt/conda/lib/python3.10/site-packages/stanza/models/tokenization/trainer.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, lambda stor

## PDF Text Extraction with BERT

BERT (Bidirectional Encoder Representations from Transformers) is a powerful language model trained on vast amounts of text data. It can predict missing words (using a technique called "masked language modeling") based on the context around them.


*   **PDF text extraction**: You are extracting titles and descriptions (or figure numbers) from a PDF document using regular expressions.

*   **Handling incomplete descriptions**: In cases where descriptions or figure titles are incomplete, BERT is used to predict the missing word. You mask the missing word with [MASK], and BERT uses the surrounding context (the words before and after) to suggest a word that would fit logically.



In [4]:
import fitz  # PyMuPDF
import re
import torch
from transformers import BertTokenizer, BertForMaskedLM

# Charger le modèle BERT
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

def extract_titles_and_descriptions_from_pdf(pdf_file):
    pdf_document = fitz.open(pdf_file)
    titles_with_descriptions = []

    for page_num in range(len(pdf_document)):
        page = pdf_document[page_num]
        page_text = page.get_text()
        page_text=page_text.replace('—','\n')
        
        matches = re.findall(r'([1-9]+(\.\d+)+)\s+(.+)', page_text)
        for match in matches:
            title = match[0]
            description = match[2]

            # Prédire la partie manquante de la description en utilisant BERT
            masked_text = f"[MASK] {description}"
            input_ids = tokenizer.encode(masked_text, add_special_tokens=True, return_tensors="pt")
            mask_index = input_ids[0].tolist().index(tokenizer.mask_token_id)

            with torch.no_grad():
                predictions = model(input_ids)
            predicted_token_id = torch.argmax(predictions.logits[0, mask_index]).item()
            predicted_word = tokenizer.decode(predicted_token_id)

            # Remplacez le masque par le mot prédit dans la description
            description = description.replace("[MASK]", predicted_word)

            # Ajoutez le titre et la description complète à la liste
            titles_with_descriptions.append(f"{title} {description}")

    pdf_document.close()
    return titles_with_descriptions

# Exemple d'utilisation
pdf_file_path = '/kaggle/input/pmi-practise/practice-standard-project-risk-management.pdf'
titles_with_descriptions = extract_titles_and_descriptions_from_pdf(pdf_file_path)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

## Project Risk Management Corpus Extraction

In this section, we used pdfplumber to extract text from specific pages (431 to 494) of the PDF file.

In [5]:
file = open('/kaggle/input/pmi-practise/practice-standard-project-risk-management.pdf','rb')
project_risk_management = ''
with pdfplumber.open(file) as pdf:
    for i in range(12,67):
        page = pdf.pages[i].filter(lambda obj: not (obj["object_type"] == "char" and obj["size"] > 30))
        project_risk_management += page.extract_text()
project_risk_management = project_risk_management.replace('\n','\n ')
project_risk_management = project_risk_management.replace('  ',' ')
project_risk_management = project_risk_management.lower()

## Fine-tuning GPT-2 for Concept Definitions

In this section, we fine-tuned a pretrained GPT-2 model on the text corpus (extracted from the Project Risk Management PDF) for generating definitions of key concepts.

In [6]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# Load the pretrained GPT-2 model and tokenizer
model_name = "gpt2-medium"  # Choose a GPT-2 variant
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(




We fine-tuned GPT-2 on our specific text: By training GPT-2 on the Project Risk Management text, it learned the language and context related to our field. This helped it generate definitions that fit the terminology we're working with.

In [7]:
file = open('output.txt','w')
file.write(project_risk_management)
file.close()

In [ ]:
# Load and preprocess your fine-tuning dataset
# Replace 'your_dataset.txt' with the path to your dataset
dataset = TextDataset(
    tokenizer=tokenizer,
    file_path='output.txt',
    block_size=128,  # Adjust block size as needed
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
)

# Configure training arguments
training_args = TrainingArguments(
    output_dir='./fine-tuned-gpt2',  # Specify the output directory
    overwrite_output_dir=True,
    num_train_epochs=500,  # Adjust the number of training epochs
    per_device_train_batch_size=16,  # Adjust batch size as needed
    save_steps=10_000,  # Specify how often to save the model
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model
trainer.save_model()

# You can now use the fine-tuned model for text generation tasks

/opt/conda/lib/python3.10/site-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Step,Training Loss
500,0.611000
1000,0.023500


In [ ]:
from joblib import dump, load

In [ ]:
# Sauvegarder le modèle au format .pt
model_path = "fine_tuned_model.pt"
torch.save(model.state_dict(), model_path)

print(f"Le modèle a été sauvegardé sous {model_path}")

We provided a list of concepts (i.e., cleaned_concept_list) and created a prompt for each concept, such as "define the [concept] concept." This prompt acts as an instruction for GPT-2, telling it to generate a definition for the given concept. The model uses this prompt to understand what kind of text (in this case, a definition) you are asking for.

In [ ]:
head_type_tail=pd.read_csv("/kaggle/input/head-type-tail/head_type_tail.csv")

In [ ]:
# import pandas as pd
# from transformers import GPT2LMHeadModel, GPT2Tokenizer

# # Step 1: Load the head-type-tail dataset (assuming it's in a DataFrame format)
# # Example DataFrame structure: columns: ['head', 'type', 'tail']
# # Replace this with your actual DataFrame


# # Step 2: Extract the heads into a list
# heads_list = head_type_tail['head'].tolist()

# # Step 3: Load the fine-tuned GPT-2 model and tokenizer
# fine_tuned_model = GPT2LMHeadModel.from_pretrained('./fine-tuned-gpt2')  # Load from the fine-tuned model directory
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2")  # Load the GPT2 tokenizer


# # Set pad_token_id to eos_token_id (to avoid padding issues)
# tokenizer.pad_token_id = tokenizer.eos_token_id
# fine_tuned_model.pad_token_id = tokenizer.eos_token_id

# # Set the model to evaluation mode
# fine_tuned_model.eval()

# # Step 4: Initialize a list to store the results (head, definition, type, tail)
# results = []

# # Step 5: Generate definitions for each head in the list
# for i, head in enumerate(heads_list):
#     # Define a starting prompt for text generation
#     prompt = f"define the {head} concept."
    
#     # Tokenize the prompt and create an attention mask
#     input_ids = tokenizer.encode(prompt, return_tensors="pt")
#     attention_mask = (input_ids != tokenizer.pad_token_id).long()  # Create attention mask

#     # Generate text using the fine-tuned model
#     output = fine_tuned_model.generate(
#         input_ids,
#         attention_mask=attention_mask,  # Pass the attention mask
#         max_length=200,
#         num_return_sequences=1,
#         pad_token_id=tokenizer.eos_token_id  # Explicitly set the pad_token_id for generation
#     )

#     # Decode the generated text
#     generated_text = tokenizer.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    
#     # Step 6: Collect the head, generated definition, type, and tail into a dictionary
#     row_data = {
#         'head': head,
#         'definition': generated_text,
#         'type': head_type_tail.loc[i, 'type'],  # Get corresponding type
#         'tail': head_type_tail.loc[i, 'tail']   # Get corresponding tail
#     }
    
#     # Append the result to the results list
#     results.append(row_data)

# # Step 7: Convert the results list into a new DataFrame
# final_df = pd.DataFrame(results)



In [ ]:
htt = final_df.copy()

In [ ]:
htt['Concept'] = htt['head']

In [ ]:
cleaned_concept_list = [' '.join(filter(str.isalpha, element.split())) for element in titles_with_descriptions]

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the fine-tuned model and tokenizer
fine_tuned_model = GPT2LMHeadModel.from_pretrained('./fine-tuned-gpt2')  # Load from the fine-tuned model directory
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")  # Load the GPT2 tokenizer

# Set pad_token_id to eos_token_id


tokenizer.pad_token_id = tokenizer.eos_token_id
fine_tuned_model.pad_token_id = tokenizer.eos_token_id

# Set the model to evaluation mode
fine_tuned_model.eval()

# Initialize the dictionary to store definitions
definitions = {}

for i in range(len(cleaned_concept_list)):

    # Define a starting prompt for text generation
    prompt = "define the " + cleaned_concept_list[i] + " concept."
    print(i)
    # Tokenize the prompt and create an attention mask
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = (input_ids != tokenizer.pad_token_id).long()  # Create attention mask

    # Generate text using the fine-tuned model
    output = fine_tuned_model.generate(
        input_ids,
        attention_mask=attention_mask,  # Pass the attention mask
        max_length=200,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id  # Explicitly set the pad_token_id for generation
    )

    # Decode the generated text with clean_up_tokenization_spaces explicitly set
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    definitions[cleaned_concept_list[i]] = generated_text

# Print or store the definitions as needed
#print(definitions)

In [ ]:
len(cleaned_concept_list)

In [ ]:
definitions

## Creating DataFrame for Concept Organization

For this section, where you're creating a DataFrame to organize the extracted concepts and other relevant information

In [ ]:
# data_frame = pd.DataFrame()
# data_frame['risk_concepts'] = [c for c in  titles_with_descriptions]
# data_frame['Relation_Type'] = ""
# data_frame['Definition'] = ""
# data_frame['Clean_definition'] = ""
# data_frame['Figure'] = ""
# data_frame['Described_in'] = ""

### Extract the Relation_Type

**How We'll Use the Features in the Knowledge Graph:**

* Concepts: These are the nodes representing key entities.
* Type_relation: Defines the edges, showing relationships between nodes.
* Concept_of_type_relation: Represents the target nodes linked by Type_relation.
* Definition: Adds attributes to nodes for additional context.
* Synonym: Merges related terms into a single node.
* Reference: Links nodes to external sources for further information.
* Process_name: Describes actions as edge labels between nodes.

Each feature helps create a structured and meaningful knowledge graph.

here we extracted the text after the colon ":" and used it to update the Relation_Type column with a corresponding relationship.

In [ ]:
for idx, row in data_frame.iterrows():
    if ":" in str(row.risk_concepts) :
            key=str(row.risk_concepts)[str(row.risk_concepts).index(":")+2:]
            data_frame.at[idx, 'Relation_Type'] = f'Has {key}'

In [ ]:
import pandas as pd

# Assuming 'definitions' is the dictionary you want to save
# Convert the dictionary to a pandas DataFrame
df = pd.DataFrame(list(definitions.items()), columns=['Concept', 'Definition'])

# Save the DataFrame to a CSV file
csv_file_path = '/kaggle/working/definitions.csv'
df.to_csv(csv_file_path, index=False)

# Provide the file path for download
csv_file_path

### Add the Definition

In [ ]:
data_df=pd.read_csv("/kaggle/input/definitions-pmi/definitions.csv")

In [ ]:
data_df = data_df.drop(index=115).reset_index(drop=True)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
data_df

In [ ]:
data_frame = data_frame.drop(index=0).reset_index(drop=True)

In [ ]:
import re

# Function to normalize the concepts
def normalize_concept(concept):
    # Convert to lowercase
    concept = concept.lower()

    # Remove any numbers, special characters, and excessive spaces
    concept = re.sub(r'[^a-z\s]', '', concept)  # Keep only letters and spaces
    concept = re.sub(r'\s+', ' ', concept).strip()  # Remove extra spaces

    return concept

# Apply the normalization to the 'Concept' column in both dataframes
data_df['Concept'] = data_df['Concept'].apply(normalize_concept)
data_frame['risk_concepts'] = data_frame['risk_concepts'].apply(normalize_concept)

# Check the updated dataframes to see the normalized concepts
# print(data_df['Concept'].head())
# print(data_frame['Concept'].head())


We **normalized** the concepts by converting them to **lowercase and removing special characters** to ensure consistency across both dataframes, making it easier to accurately match concepts between the two datasets.

In [ ]:
# Assuming data_df and risk_df are the two dataframes shown in the images
# Let's find the matching keys between the two columns of the first column (Concept and risk_concepts)

# Convert both columns to lowercase and strip whitespaces to ensure correct matching
data_df['Concept'] = data_df['Concept'].str.lower().str.strip()
data_frame['risk_concepts'] = data_frame['risk_concepts'].str.lower().str.strip()

# Find the intersection of the two columns
matching_keys = data_df[data_df['Concept'].isin(data_frame['risk_concepts'])]



In [ ]:
data_frame = matching_keys.copy()

In [ ]:
data_frame['Relation_Type'] = ""
data_frame['Clean_definition'] = ""
data_frame['Figure'] = ""
data_frame['Described_in'] = ""

In [ ]:
data_frame.rename(columns={'Concept': 'risk_concepts'}, inplace=True)

In [ ]:
htt

In [ ]:
htt.rename(columns={'head': 'Concepts'}, inplace=True)
htt.rename(columns={'type': 'Type_relation'}, inplace=True)
htt.rename(columns={'tail': 'Concept_of_type_relation'}, inplace=True)
htt.rename(columns={'definition': 'Definition'}, inplace=True)

In [ ]:
htt["Reference"] = ""

In [ ]:
htt["Figure"] = ""

In [ ]:
htt["Synonym"] = ""

### Extract the Described in section

1. Loaded the GPT-2 model and tokenizer: We used a pretrained GPT-2 model and tokenizer to process text for analyzing and extracting specific patterns.
2. Created a function to extract section numbers: We defined a function, extraire_numeros_sections, to find and extract all section numbers (formatted as "x.x.x.x") that follow the phrase "described in section" in the given text.
3. Processed text with GPT-2: For each occurrence of "described in section," we tokenized the text and passed it through the GPT-2 model for encoding.
4. Used regex to find section numbers: Finally, we applied a regular expression to extract the section numbers (e.g., "x.x.x.x") and returned them as a list, ensuring unique section numbers by storing them in a set.

In [ ]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2Model
import torch
import re



# Chargez le tokenizer GPT-2 préentraîné
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Chargez le modèle GPT-2 préentraîné
model = GPT2Model.from_pretrained("gpt2")

# Définissez une fonction pour extraire tous les numéros de section au format "x.x.x.x" après "described in section"
def extraire_numeros_sections(texte):
    # Recherchez tous les textes "described in section" dans le texte
    indices = [match.start() for match in re.finditer("described in section", texte)]

    numeros_sections = set()  # Utilisez un ensemble pour stocker les numéros de section uniques

    for start_idx in indices:
        # Extrait le texte après "described in section"
        texte_apres_section = texte[start_idx + len("described in section"):].strip()

        # Ajoutez une séquence d'arrêt de texte au texte
        texte_apres_section += tokenizer.eos_token

        # Utilisez le tokenizer pour prétraiter le texte
        texte_enc = tokenizer(texte_apres_section, return_tensors="pt")

        # Passez les données au modèle GPT-2 pour l'encodage
        with torch.no_grad():
            outputs = model(**texte_enc)

        # Utilisez une expression régulière pour extraire les numéros de section (x.x.x.x)
        numeros_section_match = re.findall(r'described in section (\d+\.\d+\.\d+\.\d+)', texte)

        if numeros_section_match:
            numeros_sections.update(numeros_section_match)  # Utilisez "update" pour ajouter des éléments à l'ensemble

    return ["descriped in section "+ str(i) for i in list(numeros_sections)]


### Definition Cleanup and Extraction

We extracted section and figure references from the Definition column, cleaned them, and updated the relevant entries.

In [ ]:
item_list = []
for item in data_frame["Definition"]:
    if isinstance(item, str):  # Check if the item is a string
        item_list.append(item.split())
    else:
        item_list.append([])  # Append an empty list or handle as needed for NaN values

item_list_not_splitted = []
for item in data_frame["Definition"]:
    item_list_not_splitted.append(item if isinstance(item, str) else None)  # Append None for NaN values


In [ ]:
# Extract
described_list = []
index_list = []
for i in range(len(item_list)):
    for j in range(len(item_list[i])):
        # Ensure there's enough room to access j+1, j+2, and j+3
        if (j + 3 < len(item_list[i])) and (
            (item_list[i][j] == "described" and item_list[i][j + 1] == "in" and item_list[i][j + 2] == "section") or
            (item_list[i][j] == "(described" and item_list[i][j + 1] == "in" and item_list[i][j + 2] == "section")
        ):
            described_list.append(item_list[i][j] + " " + item_list[i][j + 1] + " " + item_list[i][j + 2] + " " + item_list[i][j + 3])
            index_list.append(i)

# Load and remove
for idx in index_list:
    # Use .iloc[] for assignment
    data_frame['Clean_definition'].iloc[idx] = item_list_not_splitted[idx].replace("described in section", "").replace("section", "").replace("figure", "")


In [ ]:
# #Extract
# figure_list = []
# figure_index_list = []
# for i in range(len(item_list)):
#     for j in range(len(item_list[i])):
#         if item_list[i][j] == "figure":
#             figure_list.append(item_list[i][j]+" "+item_list[i][j+1])
#             #print(item_list[i][j+1])
#             figure_index_list.append(i)

# #load and remove
# for i in range(len(figure_index_list)):
#     data_frame['Figure'].iloc[figure_index_list[i]] += " " + (figure_list[i])

In [ ]:
# definition_row = data_frame.loc[210, 'Clean_definition']
# print(definition_row)

## Data Pre-processing

In [ ]:
def remove_footers(text):
    
    cleaned_text = re.sub(r'(©2009 project management institute\. practice standard for project risk management)', '', text)
    cleaned_text = re.sub(r'\s{2,}', ' ', cleaned_text)
    cleaned_text = cleaned_text.strip()
    return cleaned_text

In [ ]:
data_frame['Clean_definition'] = data_df['Definition'].apply(remove_footers)

In [ ]:
#Extract
figure_list = []
figure_index_list = []
for i in range(len(item_list)):
    for j in range(len(item_list[i])):
        if item_list[i][j] == "figure":
            figure_list.append(item_list[i][j]+" "+item_list[i][j+1])
            #print(item_list[i][j+1])
            figure_index_list.append(i)

#load and remove
for i in range(len(figure_index_list)):
    data_frame['Figure'].iloc[figure_index_list[i]] += " " + (figure_list[i])

In [ ]:
for i in range(len(figure_index_list)):
    data_frame['Clean_definition'].iloc[figure_index_list[i]] = item_list_not_splitted[figure_index_list[i]].replace("described in section", "")
    data_frame['Clean_definition'].iloc[figure_index_list[i]] = item_list_not_splitted[figure_index_list[i]].replace("section","")
    data_frame['Clean_definition'].iloc[figure_index_list[i]] = item_list_not_splitted[figure_index_list[i]].replace("figure","")

In [ ]:
import re

# Function to remove section numbers like "11.2.1.2"
def remove_section_numbers(text):
    # Regular expression to match patterns like 11.2.1.2, 4.2.3.4, etc.
    text = re.sub(r'\b\d+(\.\d+)+\b', '', text)

    # Remove extra spaces left after removal of numbers
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
htt['Definition'] = htt['Definition'].apply(remove_section_numbers)

In [ ]:
# Apply the function to the 'Clean_definition' column
data_frame['Clean_definition'] = data_frame['Clean_definition'].apply(remove_section_numbers)

In [ ]:
def remove_headers(text):
    
    cleaned_text = re.sub(r'chapter\s+\d+\s*(−|-|:)?\s*[A-Z\s]+', '', text, flags=re.IGNORECASE)
    
    
    cleaned_text = re.sub(r'chapter\s+[IVXLCDM]+\s*(−|-|:)?\s*[A-Z\s]+', '', cleaned_text, flags=re.IGNORECASE)
    
    
    cleaned_text = re.sub(r'\s{2,}', ' ', cleaned_text).strip()
    
    return cleaned_text

In [ ]:
def remove_special_characters(text_original):
    return re.sub(r'[^A-Za-z0-9\s\.]', '', text_original)

In [ ]:
def remove_single_letter_words(text):

    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
def preprocess_text(text):
    """Preprocess text by applying various cleaning steps.""" 
    
    #text = remove_footers(text)
    text = remove_headers(text)
    text = re.sub(r'\d+', '', text)
    
    text = remove_special_characters(text)        
    text = re.sub(r'\.{2,}', '.', text)
    
    text = ' '.join(text.split())
    
    text = re.sub(r'\s+', ' ', text)

    
    text = re.sub(r'\. *\.', '.', text)
    
    text = re.sub(r'(?<!\w)\. +(?=[A-Z])', '', text)  
    
    text = re.sub(r'^\.+', '', text)
    remove_single_letter_words(text)

    return text

In [ ]:
data_frame["Clean_definition"]=data_frame["Clean_definition"].apply(lambda s : s.lower())

In [ ]:
htt["Definition"]=htt["Definition"].apply(lambda s : s.lower())

In [ ]:
data_frame["Clean_definition"]=data_frame["Clean_definition"].apply(preprocess_text)

In [ ]:
htt["Definition"]=htt["Definition"].apply(preprocess_text)

In [ ]:
data_frame = data_frame.drop('Definition', axis=1)

## Extracting and Storing Triplet Relations from Definitions

In [6]:
data_frame=pd.read_csv("/kaggle/input/first-version/first_version.csv")

In [7]:
data_frame['Type_relation'] = data_frame['Type_relation'].fillna('')
data_frame['Concept_of_type_relation'] = data_frame['Concept_of_type_relation'].fillna('')
data_frame['Synonym'] = data_frame['Synonym'].fillna('')
data_frame['Process_name'] = data_frame['Process_name'].fillna('')
data_frame['Figure'] = data_frame['Figure'].fillna('')

In [8]:
data_frame

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure
0,pmi purpose,,,define the purpose of the practice standard fo...,,[' 1.1 purpose of the practice standard for pr...,,figure 1-1.
1,project risk management definition,,,define the project risk management de nition c...,,[' 1.2 project risk management defi nition'],,
2,project risk management in project management ...,,,define the role of project risk management in ...,,[' 1.3 role of project risk management in proj...,,
3,good risk management practice,,,define the good risk management practice conce...,,[' plan risk management'],,
4,project risk management success factors,,,define the critical success factors for projec...,,[' 1.5 critical success factors for project ri...,,
...,...,...,...,...,...,...,...,...
144,risk audits,,,define the risk audits concept. t he term risk...,,[' perform quantitative risk analysis'],,
145,risk reassessment,,,define the risk reassessment concept. t he pro...,,[' risk reassessment are:'],,
146,status meetings,,,define the status meetings concept. t he proje...,,[' analysis process'],,
147,trend analysis,,,define the trend analysis concept. t he term p...,,[' analysis process'],,


### Creating the Triplets DataFrame

What we did: We initialized an empty DataFrame called Triplets with columns for risk_concepts, head, type, and tail.
Why: This DataFrame will be used to store the extracted relational triplets (subject, relation, object) for each risk concept, making it easier to manage and analyze later.

In [9]:
Triplets = pd.DataFrame()
Triplets['risk_concepts'] = data_frame['Concepts']
Triplets['head'] = ""
Triplets['type'] = ""
Triplets['tail'] = ""

### Loading a Pretrained Model (REBEL)

What we did: We loaded the **REBEL** model and tokenizer, which are designed for extracting relational triplets.
Why: The REBEL(Relation Extraction By End-to-end Language generation) model is specialized for relation extraction tasks, which makes it ideal for identifying and generating triplet relationships (subject, relation, object) from text in our definitions.

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [11]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Extracting Relations from Model Output

What we did: We defined the extract_relations_from_model_output() function to parse the model's output and organize the extracted triplets.
Why: This function helps break down the model’s predictions into structured triplets (head, type, tail) so we can store them in a clean and organized manner.

In [12]:
def extract_relations_from_model_output(text):
    relations = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    text_replaced = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "")
    for token in text_replaced.split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        relations.append({
            'head': subject.strip(),
            'type': relation.strip(),
            'tail': object_.strip()
        })
    return relations

### A Knowledge Base (KB) Class

What we did: We built a KB class to store relations, check for duplicates, and add new unique relations.
Why: We needed a way to manage relations and avoid duplicates. The KB class ensures that only unique triplets are stored, making the data cleaner and easier to work with.

In [13]:
class KB():
    def __init__(self):
        self.relations = []

    def are_relations_equal(self, r1, r2):
        return all(r1[attr] == r2[attr] for attr in ["head", "type", "tail"])

    def exists_relation(self, r1):
        return any(self.are_relations_equal(r1, r2) for r2 in self.relations)

    def add_relation(self, r):
        if not self.exists_relation(r):
            self.relations.append(r)

    def save(self):
        dict_list = []
        for r in self.relations:
            dict_list.append(r)
        return(dict_list)
        #df = pd.DataFrame(dict_list, columns=['head', 'type', 'tail'])
        #print(dict_list)

### Processing Text into Relations

In [14]:
def from_small_text_to_kb(text, verbose=False):
    kb = KB()

    # Tokenizer text
    model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True,
                            return_tensors='pt')
    if verbose:
        print(f"Num tokens: {len(model_inputs['input_ids'][0])}")

    # Generate
    gen_kwargs = {
        "max_length": 1000,
        "length_penalty": 0,
        "num_beams": 3,
        "num_return_sequences": 3
    }

    generated_tokens = model.generate(
        **model_inputs,
        **gen_kwargs,
    )

    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

    # create kb
    for sentence_pred in decoded_preds:
        relations = extract_relations_from_model_output(sentence_pred)
        for r in relations:
            kb.add_relation(r)

    return kb

In [15]:
from tqdm import tqdm

In [16]:
relation_list =[]
for item in tqdm(data_frame['Definition']):
    kb = from_small_text_to_kb(item, verbose=True)
    relation_list.append(kb.save())

  0%|          | 0/149 [00:00<?, ?it/s]

Num tokens: 177


  1%|          | 1/149 [00:01<03:16,  1.33s/it]

Num tokens: 169


  1%|▏         | 2/149 [00:03<04:22,  1.78s/it]

...

 99%|█████████▊| 147/149 [03:58<00:02,  1.45s/it]

Num tokens: 169


 99%|█████████▉| 148/149 [03:59<00:01,  1.38s/it]

Num tokens: 139


100%|██████████| 149/149 [04:01<00:00,  1.62s/it]


In [17]:
dict_list =[]
for item in tqdm(relation_list):
    for dic in item:
        dict_list.append(dic)
triplet = pd.DataFrame(dict_list, columns=['head', 'type', 'tail'])

100%|██████████| 149/149 [00:00<00:00, 338177.11it/s]


We did this to organize all the triplet relations (subject, relation, object) into a simple and structured table, so it's easy to work with and analyze.

In [18]:
unique_triplets = triplet.drop_duplicates()

In [19]:
tiplet_unique = triplet['head'].unique()

In [20]:
final_triplet_unique = triplet.drop_duplicates(keep='first')

In [21]:
from fuzzywuzzy import fuzz

def are_triplets_similar(row1, row2, threshold=80):
    # Compute fuzzy similarity scores for 'head', 'type', and 'tail'
    head_similarity = fuzz.partial_ratio(row1['head'].lower(), row2['head'].lower())
    type_similarity = fuzz.partial_ratio(row1['type'].lower(), row2['type'].lower())
    tail_similarity = fuzz.partial_ratio(row1['tail'].lower(), row2['tail'].lower())
    
    # Check if the average similarity exceeds the threshold
    average_similarity = (head_similarity + type_similarity + tail_similarity) / 3
    
    return average_similarity > threshold

# Function to remove similar triplets
def remove_similar_triplets(triplet_df, threshold=80):
    rows_to_remove = set()
    
    # Loop over each row in the DataFrame
    for i in range(len(triplet_df)):
        if i in rows_to_remove:
            continue
        
        for j in range(i + 1, len(triplet_df)):
            # Check if the two triplets are semantically similar
            if j not in rows_to_remove and are_triplets_similar(triplet_df.iloc[i], triplet_df.iloc[j], threshold):
                rows_to_remove.add(j)

    # Drop rows that were marked as similar
    cleaned_df = triplet_df.drop(rows_to_remove).reset_index(drop=True)
    return cleaned_df

/opt/conda/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [22]:
final_triplet_unique.reset_index(drop=True, inplace=True)

In [23]:
cleaned_triplet_df = remove_similar_triplets(final_triplet_unique, threshold=80)

### Removing Stop Words

In [24]:
!python -m spacy download en_core_web_lg

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 3.0 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [25]:
import textacy
import nltk
from nltk.corpus import stopwords
import copy

# Download the stopwords dataset (if not already downloaded)
nltk.download('stopwords')
from nltk.tokenize import word_tokenize, sent_tokenize
import spacy
nlp_spacy = spacy.load("en_core_web_lg")


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [26]:
text = ''
for item in data_frame['Definition']:
    text += ''.join(item)

1. Loaded stop words
2. Tokenized the text
3. Filtered out stop words
4. Reconstructed the sentence

In [27]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary data
nltk.download('punkt')
nltk.download('stopwords')

example_sent = text
stop_words = set(stopwords.words('english'))
word_tokens = word_tokenize(example_sent)

# Filter out the stop words
filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]

# Join the filtered words back into a sentence
test_text = ' '.join(filtered_sentence)




[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [28]:
import textacy
triple_list = []
for sentence in test_text.split('.'):
    t1 = nlp_spacy(sentence)
    triple = textacy.extract.subject_verb_object_triples(t1)
    if triple:
        triple_to_list = list(triple)
        triple_list.append(pd.DataFrame(triple_to_list))
svo=pd.concat(triple_list, axis=0) # this should concat all dfs on top of one another using axis=0
svo.columns=['subject','verb','object'] # change your columns on teh final df

We extracted and organized subject-verb-object (SVO) triples to better understand relationships within the text and to analyze how subjects, actions (verbs), and objects are related in each sentence.

In [29]:
svo = svo.reset_index(drop=True)  # Reset the index to have unique values

### Converting spaCy Tokens to Text in SVO DataFrame

In [30]:
final_svo = copy.deepcopy(svo)

In [31]:
for idx,row in final_svo.iterrows():
    for c in final_svo.columns:
        for i in row[c]:
            if isinstance(i, spacy.tokens.span.Span):
                row[c] = i.text
            else:
                row[c] = i

In [32]:
for idx,item in enumerate(final_svo['subject']):
    if isinstance(item, spacy.tokens.token.Token):
        item = item.text
        final_svo.iloc[idx]['subject'] = item
    else:
        item = item
        final_svo.iloc[idx]['subject'] = item
for idx,item in enumerate(final_svo['verb']):
    if isinstance(item, spacy.tokens.token.Token):
        item = item.text
        final_svo.iloc[idx]['verb'] = item
    else:
        item = item
        final_svo.iloc[idx]['verb'] = item
for idx,item in enumerate(final_svo['object']):
    if isinstance(item, spacy.tokens.token.Token):
        item = item.text
        final_svo.iloc[idx]['object'] = item
    else:
        item = item
        final_svo.iloc[idx]['object'] = item

/tmp/ipykernel_398/2970029063.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  final_svo.iloc[idx]['subject'] = item
/tmp/ipykernel_398/2970029063.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are 

Identifying and Storing Unique Concepts from SVO

In [33]:
concept_list = []
concept_index = []
for _,row in final_svo.iterrows():
    item = row['subject']
    for i in tiplet_unique:
        if (i in item):
            concept_list.append(item)
            concept_index.append(list(final_svo['subject']).index(item))
svo_unique = dict(zip(concept_index, concept_list))

In [34]:
final_svo_unique = final_svo.drop_duplicates(keep='first')

## Final Data Frame

In [35]:
data_final_df = data_frame.copy()

In [ ]:
# df_final = pd.DataFrame()
# df_final['Concepts'] = data_frame['risk_concepts']
# df_final['Type_relation'] = ''
# df_final['Concept_of_type_relation'] = ''
# df_final['Definition'] = data_frame['Clean_definition']
# df_final['Synonym'] = ''
# df_final['Reference'] = data_frame['Described_in']
# df_final['Process_name'] = ''
# df_final['Figure'] = data_frame['Figure']

### References

In [ ]:
# import fitz  # PyMuPDF to extract text from PDF
# import torch
# from transformers import BertTokenizer, BertModel
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np

# # Check if GPU is available
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Load a pretrained BERT model and tokenizer
# model_name = 'bert-base-uncased'
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertModel.from_pretrained(model_name).to(device)  # Move model to GPU if available

# def get_bert_embeddings(text):
#     """
#     Function to generate BERT embeddings for a given text
#     """
#     inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True).to(device)  # Move inputs to GPU
#     with torch.no_grad():
#         outputs = model(**inputs)
#     return outputs.last_hidden_state.mean(dim=1).squeeze().cpu()  # Move result back to CPU

# def find_concept_in_document(concept, document_text):
#     """
#     Search where the concept is defined in the document using BERT embeddings for similarity comparison
#     """
#     sections = document_text.split("\n")  # Split the document into paragraphs or sections
#     concept_embedding = get_bert_embeddings(concept)

#     most_similar_section = None
#     max_similarity = -1

#     # Compare each section with the concept definition using cosine similarity
#     for section in sections:
#         section_embedding = get_bert_embeddings(section)
#         similarity = cosine_similarity([concept_embedding], [section_embedding])[0][0]

#         if similarity > max_similarity:
#             max_similarity = similarity
#             most_similar_section = section
    
#     return most_similar_section, max_similarity

# def update_dataset_with_references(df, document_text):
#     """
#     For each concept in the dataset, find its definition in the document and update the dataset
#     """
#     for idx, row in df.iterrows():
#         concept = row['Concepts']
#         definition_section, similarity = find_concept_in_document(concept, document_text)
        
#         if definition_section:
#             # You can update the 'Reference' or 'Process_name' with the most relevant section
#             df.at[idx, 'Reference'] = [definition_section]  # Add the section where it's defined
            
#     return df

# # Assuming df_final is your dataset with concepts and definitions
# data_final_df = update_dataset_with_references(data_final_df, project_risk_management)


### Simplify Concept Name

In [ ]:
# def simplify_concept_name(concept):
#     simplified_concept = concept  # Initialize simplified_concept with the original concept
    
#     # Use regular expression to reverse subject and object in concepts with "of" or "for"
#     if 'of' in concept:
#         match = re.match(r'(.*?) of (.*)', concept)
#         if match:
#             qualifier, subject = match.groups()
#             simplified_concept = f'{subject.strip()} {qualifier.strip()}'
#     elif 'for' in concept:
#         match = re.match(r'(.*?) for (.*)', concept)
#         if match:
#             qualifier, subject = match.groups()
#             simplified_concept = f'{subject.strip()} {qualifier.strip()}'
    
#     # Remove any occurrence of "the" from the simplified concept
#     simplified_concept = re.sub(r'\bthe\b', '', simplified_concept, flags=re.IGNORECASE).strip()
    
#     simplified_concept = re.sub(r'\bdocument(ing)?\b', '', simplified_concept, flags=re.IGNORECASE).strip()
    
#     simplified_concept = re.sub(r'\bcritical\b', '', simplified_concept, flags=re.IGNORECASE).strip()
    
#     simplified_concept = re.sub(r'\bpractice standard for project risk management\b', 'pmi', simplified_concept, flags=re.IGNORECASE).strip()
    
    
   
#     if "purpose" in simplified_concept and "objectives" in simplified_concept:
#         simplified_concept = re.sub(r'\bpurpose and objectives\b', 'objectives', simplified_concept, flags=re.IGNORECASE)
    
#     if "techniques" in simplified_concept and "tools and techniques" not in simplified_concept:
#         simplified_concept = re.sub(r'\btechniques\b', 'tools and techniques', simplified_concept, flags=re.IGNORECASE)
        
#     simplified_concept = re.sub(r'\s+', ' ', simplified_concept).strip()


#     return simplified_concept

We wanted to connect related information from final_triplet_unique to df_final, so we added the relationship details and corresponding concepts when they matched.

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.metrics.pairwise import cosine_similarity
# from fuzzywuzzy import fuzz
# from transformers import BertTokenizer, BertModel
# import torch

# # Load BERT tokenizer and model for embeddings
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# model = BertModel.from_pretrained('bert-base-uncased')

# # Function to get BERT embeddings for text
# def get_bert_embedding(text):
#     inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
#     outputs = model(**inputs)
#     return outputs.last_hidden_state.mean(dim=1).detach().numpy()

# # Function to match triplet heads to concepts based on their definitions
# def match_triplet_to_concept(triplets_df, concepts_df):
#     triplet_heads = triplets_df['head'].tolist()  # List of triplet heads
#     triplet_tails = triplets_df['tail'].tolist()  # List of triplet tails
#     triplet_types = triplets_df['type'].tolist()  # List of relation types
#     concept_definitions = concepts_df['Definition'].tolist()  # List of concept definitions

#     # Initialize new columns in concepts_df with default values
#     concepts_df['Type_relation'] = "No relation found"  
#     concepts_df['Concept_of_type_relation'] = "No concept found"

#     # Iterate over each definition and find the most frequent matching head
#     for j, definition in enumerate(concept_definitions):
#         if pd.isna(definition) or definition.strip() == "":
#             print(f"Definition at index {j} is empty or NaN. Skipping.")
#             continue  # Skip if the definition is NaN or empty

#         max_count = 0
#         best_match = None
#         best_type = None
#         best_tail = None

#         # Check direct, token, and fuzzy matches for each triplet head
#         for i, head in enumerate(triplet_heads):
#             # Direct match
#             if head in definition:
#                 count = definition.count(head)  # Count occurrences of the head in the definition
#             else:
#                 # Token-based matching (split head into tokens and match them)
#                 head_tokens = head.split()  # Split the head into tokens
#                 count = sum(definition.count(token) for token in head_tokens)  # Count token matches

#             # Fuzzy matching
#             fuzz_ratio = fuzz.ratio(head.lower(), definition.lower())
#             if fuzz_ratio > 70:  # Adjust threshold as necessary
#                 count += 1  # Increment count if there's a good fuzzy match

#             # Update best match if current count is higher
#             if count > max_count:
#                 max_count = count
#                 best_match = triplet_heads[i]
#                 best_type = triplet_types[i]
#                 best_tail = triplet_tails[i]

#         # Assign the most frequent head and corresponding relation and tail
#         if best_match:
#             concepts_df.at[j, 'Type_relation'] = best_type
#             concepts_df.at[j, 'Concept_of_type_relation'] = best_tail
#         else:
#             # If no match is found, fallback to semantic similarity
#             print(f"No direct match found for definition at index {j}. Falling back to semantic similarity.")
#             concepts_df = semantic_fallback(concepts_df, triplet_heads, concept_definitions, j, triplet_types, triplet_tails)

#     # After processing, check for any NaN values in the new columns and handle them
#     concepts_df['Type_relation'].replace("No relation found", np.nan, inplace=True)
#     concepts_df['Concept_of_type_relation'].replace("No concept found", np.nan, inplace=True)

#     return concepts_df

# # Function to calculate semantic similarity using cosine similarity for fallback
# def semantic_fallback(concepts_df, triplet_heads, concept_definitions, definition_idx, triplet_types, triplet_tails):
#     # Get the current concept's definition
#     current_definition = concept_definitions[definition_idx]

#     # Get embeddings for the definition and heads (using BERT or a similar model)
#     definition_embedding = get_bert_embedding(current_definition)
#     head_embeddings = [get_bert_embedding(head) for head in triplet_heads]

#     # Compute cosine similarity between the definition and all triplet heads
#     similarities = cosine_similarity([definition_embedding], head_embeddings)[0]
    
#     # Find the index of the most similar head
#     best_match_idx = np.argmax(similarities)

#     # Check if the best match index is valid and not resulting in NaN
#     if similarities[best_match_idx] > 0:  # Only proceed if there's a meaningful match
#         concepts_df.at[definition_idx, 'Type_relation'] = triplet_types[best_match_idx]
#         concepts_df.at[definition_idx, 'Concept_of_type_relation'] = triplet_tails[best_match_idx]
#     else:
#         print(f"No valid semantic match found for definition at index {definition_idx}. Keeping previous values.")

#     return concepts_df

# # Example usage:
# # triplets_df should contain 'head', 'type', and 'tail' columns (for triplets like head -> relation -> tail)
# # concepts_df should contain 'Concepts', 'Definition', 'Type_relation', 'Concept_of_type_relation'.


We used stemming to simplify verbs and combined them with objects to create concise and meaningful process names, which were then added to the corresponding rows in df_final.

In [36]:
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [37]:
import torch

## Risk Management Glossary

In [38]:
import fitz  # PyMuPDF

def extract_bold_terms(pdf_path):
    # Open the PDF file
    doc = fitz.open(pdf_path)

    bold_terms = []

    # Iterate through each page
    for page_num in range(len(doc)):
        page = doc[page_num]
        blocks = page.get_text("dict")['blocks']  # Get page text in dictionary format

        for block in blocks:
            if 'lines' in block:
                for line in block['lines']:
                    for span in line['spans']:
                        # Check if the text is bold
                        if span['flags'] == 20:  # This indicates bold text
                            bold_terms.append(span['text'])

    return bold_terms

# Specify your PDF path
pdf_path = '/kaggle/input/glossary/Glossary_Of_Risk_Management.pdf'

# Extract bold terms
bold_terms = extract_bold_terms(pdf_path)

In [39]:
glossary_df = pd.DataFrame(bold_terms, columns=['bold_terms'])

In [40]:
# Cleaning the DataFrame
glossary_df['bold_terms'] = glossary_df['bold_terms'].str.replace(r'[.]', '', regex=True)  # Remove periods
glossary_df['bold_terms'] = glossary_df['bold_terms'].str.lower()  # Convert to lowercase
glossary_df = glossary_df[glossary_df['bold_terms'].str.len() > 1]  # Remove single-letter rows
glossary_df = glossary_df.reset_index(drop=True)  # Reset the index

In [41]:
import pandas as pd
from fuzzywuzzy import fuzz

# Strip any leading/trailing spaces in the relevant columns
glossary_df['bold_terms'] = glossary_df['bold_terms'].str.strip()
data_final_df['Definition'] = data_final_df['Definition'].fillna('')  # Ensure no NaN in Definitions

# Filter out glossary terms that are single words or in the exclusion list
filtered_glossary_terms = [
    term for term in glossary_df['bold_terms'] 
    if (len(term.split()) >= 1) and  # Exclude single-word terms
       term.strip().lower()   # Exclude specific terms (case-insensitive)
]

# Function to compute a weighted similarity score based on multiple fields
def compute_weighted_similarity(concept, definition, concept_of_type_relation, glossary_term, weights):
    # Compute individual similarity scores
    concept_similarity = fuzz.partial_ratio(concept.lower(), glossary_term.lower())
    definition_similarity = fuzz.partial_ratio(definition.lower(), glossary_term.lower())
    concept_of_type_relation_similarity = fuzz.partial_ratio(concept_of_type_relation.lower(), glossary_term.lower())

    # Calculate a weighted average of the similarity scores
    total_similarity = (concept_similarity * weights['concept'] +
                        definition_similarity * weights['definition'] +
                        concept_of_type_relation_similarity * weights['concept_of_type_relation']) / sum(weights.values())

    return total_similarity

# Function to find the top 3 most similar glossary terms based on multiple fields
def find_best_glossary_terms(concept, definition, concept_of_type_relation, glossary_terms, primary_threshold=80, secondary_threshold=60, weights=None, top_n=3):
    if weights is None:
        weights = {'concept': 0.3, 'definition': 0.5, 'concept_of_type_relation': 0.2}  # Default weights

    best_matches = []  # Store potential matches

    # Check for exact matches first
    for term in glossary_terms:
        if term.lower() in definition.lower():
            best_matches.append((term, 100))  # Assign max similarity for exact matches

    # Iterate through glossary terms and compute weighted similarity
    for term in glossary_terms:
        similarity = compute_weighted_similarity(concept, definition, concept_of_type_relation, term, weights)

        # Add to matches if it exceeds the primary threshold
        if similarity >= primary_threshold:
            best_matches.append((term, similarity))

    # If no best matches were found with the primary threshold, check for secondary threshold
    if not best_matches:
        for term in glossary_terms:
            similarity = compute_weighted_similarity(concept, definition, concept_of_type_relation, term, weights)
            if similarity >= secondary_threshold:
                best_matches.append((term, similarity))

    # Sort matches based on similarity score in descending order
    best_matches.sort(key=lambda x: x[1], reverse=True)

    # Return up to top N best matches or fewer if there aren't enough
    return [match[0] for match in best_matches[:top_n]]

# Applying the function to update the 'Synonym' column with up to 3 best matches
for idx, row in data_final_df.iterrows():
    concept = row['Concepts']
    definition = row['Definition']
    concept_of_type_relation = row['Concept_of_type_relation']

    # Find the top 3 best matching glossary terms considering multiple fields
    best_matches = find_best_glossary_terms(concept, definition, concept_of_type_relation, filtered_glossary_terms, top_n=5)

    # Update the 'Synonym' column with the best matches joined by commas
    data_final_df.at[idx, 'Synonym'] = ', '.join(best_matches) if best_matches else None




In [44]:
data_final_df

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure
0,pmi purpose,,,define the purpose of the practice standard fo...,"budget, process, project, project management, ...",[' 1.1 purpose of the practice standard for pr...,,figure 1-1.
1,project risk management definition,,,define the project risk management de nition c...,"objective, process, project, project managemen...",[' 1.2 project risk management defi nition'],,
2,project risk management in project management ...,,,define the role of project risk management in ...,"identify risks, plan risk management, process,...",[' 1.3 role of project risk management in proj...,,
3,good risk management practice,,,define the good risk management practice conce...,"objective, process, product, project, risk",[' plan risk management'],,
4,project risk management success factors,,,define the critical success factors for projec...,"project, project risk management, risk, standard",[' 1.5 critical success factors for project ri...,,
...,...,...,...,...,...,...,...,...
144,risk audits,,,define the risk audits concept. t he term risk...,"objective, process, project, risk, risk audits",[' perform quantitative risk analysis'],,
145,risk reassessment,,,define the risk reassessment concept. t he pro...,"execute, objective, process, project, risk",[' risk reassessment are:'],,
146,status meetings,,,define the status meetings concept. t he proje...,"identify risks, monitor, project, risk, risk m...",[' analysis process'],,
147,trend analysis,,,define the trend analysis concept. t he term p...,"objective, process, project, risk, standard",[' analysis process'],,


In [42]:
htt=pd.read_csv("/kaggle/input/triplets-dataset/htt.csv")

In [43]:
htt['Figure'] = htt['Figure'].fillna('')
htt['Process_name'] = htt['Process_name'].fillna('')

In [ ]:
htt

In [ ]:
pd.reset_option('display.max_rows', None)
pd.reset_option('display.max_colwidth', None)

In [ ]:
# import pandas as pd
# from fuzzywuzzy import fuzz
# from sklearn.metrics.pairwise import cosine_similarity
# import torch
# from transformers import BertTokenizer, BertModel

# # Load a pretrained BERT model and tokenizer
# model_name = 'bert-base-uncased'
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertModel.from_pretrained(model_name)

# # Predefined specific concepts to match
# specific_concepts = [
    
#     "plan risk management",
#     "identify risks",
#     "perform qualitative risk analysis",
#     "perform quantitative risk analysis",
#     "plan risk responses",
#     "monitor and control risks"
# ]

# # Get BERT embeddings for text
# def get_bert_embeddings(text):
#     inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
#     with torch.no_grad():
#         outputs = model(**inputs)
#     return outputs.last_hidden_state.mean(dim=1).squeeze()

# # Refined matching function using token set ratio for fuzzy matching
# def find_best_matching_concept(definition, specific_concepts, threshold=70, use_bert=False):
#     best_match = None
#     highest_similarity = 0

#     # Try to match using fuzzy token set ratio for better word-level matching
#     for concept in specific_concepts:
#         fuzzy_ratio = fuzz.token_set_ratio(definition.lower(), concept.lower())  # Token-based ratio for word-level comparison
#         if fuzzy_ratio >= threshold and fuzzy_ratio > highest_similarity:
#             highest_similarity = fuzzy_ratio
#             best_match = concept

#     # Optionally, use BERT embeddings for semantic similarity (if fuzzy matching is insufficient)
#     if use_bert and (not best_match or highest_similarity < threshold):
#         definition_embedding = get_bert_embeddings(definition)
#         max_cosine_sim = -1
#         for concept in specific_concepts:
#             concept_embedding = get_bert_embeddings(concept)
#             cosine_sim = cosine_similarity([definition_embedding], [concept_embedding])[0][0]
#             if cosine_sim > max_cosine_sim:
#                 max_cosine_sim = cosine_sim
#                 best_match = concept

#     return best_match

# # Function to iterate through the 'Definition' column and update 'Concept_of_type_relation'
# def update_concept_relations(df, specific_concepts, use_bert=False):
#     for idx, row in df.iterrows():
#         definition = row['Definition']

#         # Find the best matching concept for each definition
#         best_match = find_best_matching_concept(definition, specific_concepts, use_bert=use_bert)

#         # Update the 'Concept_of_type_relation' column with the best matching concept
#         if best_match:
#             df.at[idx, 'Concept_of_type_relation'] = best_match

#     return df

In [45]:
# Assuming df_final and final_svo_unique are already loaded DataFrames
# Sample code for loading if necessary:
# df_final = pd.read_csv('/path/to/data_final.csv')
# final_svo_unique = pd.read_csv('/path/to/final_svo_unique.csv')

# Function to check if any of the subject, verb, or object words match a Concept
def match_svo_with_concept(concept, svo_row):
    concept_words = concept.lower().split()  # Split the concept into words
    # Check if any word from subject, verb, or object matches the concept
    if any(word in concept_words for word in svo_row[['subject', 'verb', 'object']].str.lower()):
        # If match found, return the concatenated SVO terms as the process name
        return f"{svo_row['subject']} {svo_row['verb']} {svo_row['object']}"
    return None

# Iterate through df_final and apply matching logic
def find_process_name(concept, svo_df):
    for idx, svo_row in svo_df.iterrows():
        process_name = match_svo_with_concept(concept, svo_row)
        if process_name:
            return process_name  # Return the first matching SVO found
    return None

# Apply the function to fill the Process_name column in df_final
data_final_df['Process_name'] = data_final_df['Concepts'].apply(lambda concept: find_process_name(concept, final_svo_unique))


In [48]:
htt

,Concepts,Definition,Type_relation,Concept_of_type_relation,Reference,Figure,Synonym,Process_name
0,project risk management,define the project risk management concept. t ...,facet of,project management,[' plan risk management'],,"budget, objective, project, project risk manag...",management recognized projects
1,project risk management,define the project risk management concept. t ...,practiced by,project management practitioners,[' plan risk management'],,"budget, objective, project, project risk manag...",management recognized projects
2,risk management,define the risk management concept. t he defi ...,studies,manage risk,[' plan risk management'],,"identify risks, objective, process, project, p...",management recognized projects
3,project management,define the project management concept. then th...,has part,project management process,[' plan risk management'],,"objective, process, project, project managemen...",management recognized projects
4,project management plan,define the project management plan concept. t ...,use,project management,[' management plan.'],,"budget, execute, process, project, project man...",management recognized projects
...,...,...,...,...,...,...,...,...
62,monte carlo simulation,define the monte carlo simulation concept. the...,uses,probability distribution,[' perform quantitative risk analysis'],,"objective, process, project, project life cycl...",None
63,realism,define the realism concept. t he defi nition o...,facet of,realism,[' introduction'],,"objective, output, process, project, project m...",None
64,critical chain,define the critical chain concept. a ll the cr...,subclass of,project management,[' analysis process'],,"objective, perform qualitative risk analysis, ...",None
65,monitor and control risks,define the monitor and control risks concept. ...,subclass of,manage risk,[' monitor and control risks'],,"monitor, process, project, project risk manage...",stakeholders gain risks


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [87]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from fuzzywuzzy import fuzz
from collections import defaultdict

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to get BERT embeddings for text
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

# Function to calculate cosine similarity between two embeddings
def calculate_similarity(embedding1, embedding2):
    return cosine_similarity(embedding1, embedding2)[0][0]

# Function to find synonyms and semantically close terms (heads or tails)
def get_related_terms(term, similarity_threshold=0.75):
    related_terms = defaultdict(list)
    term_embedding = get_bert_embedding(term)
    
    # Iterate over heads and tails, check similarity
    for potential_term in triplets_df['head'].tolist() + triplets_df['tail'].tolist():
        if potential_term != term:
            potential_term_embedding = get_bert_embedding(potential_term)
            similarity_score = calculate_similarity(term_embedding, potential_term_embedding)
            
            if similarity_score >= similarity_threshold:
                related_terms[term].append((potential_term, similarity_score))
    
    return related_terms

# Function to match triplet heads/tails to concept definitions
def update_concept_relations(triplets_df, concepts_df, similarity_threshold=0.8, fuzzy_threshold=70):
    # Initialize new columns in concepts_df with default values
    concepts_df['Type_relation'] = ""
    concepts_df['Concept_of_type_relation'] = ""
    
    exclude_terms = ["project risk management", "risk management", "risk", "critical success factor", "project management"]

    # Iterate over each concept definition in concepts_df
    for j, concept_row in concepts_df.iterrows():
        concept_definition = concept_row['Definition']

        if pd.isna(concept_definition) or concept_definition.strip() == "":
            continue  # Skip if the definition is NaN or empty

        max_similarity = 0
        best_match = None
        best_relation = None

        # Iterate over each triplet (head, type, tail) in the triplets_df
        for i, row in triplets_df.iterrows():
            head = row['head']
            tail = row['tail']
            triplet_type = row['type']

            if head in exclude_terms or tail in exclude_terms:
                continue  # Skip this iteration if any excluded terms are found

            # Step 1: Exact match with head or tail
            if head in concept_definition or tail in concept_definition:
                best_match = head if head in concept_definition else tail
                best_relation = triplet_type
                break  # Direct match found, no need to continue for this definition

            # Step 2: Fuzzy match with head or tail (using fuzzy matching to handle partial or inexact matches)
            fuzz_head_score = fuzz.partial_ratio(head.lower(), concept_definition.lower())
            fuzz_tail_score = fuzz.partial_ratio(tail.lower(), concept_definition.lower())

            if fuzz_head_score > fuzzy_threshold or fuzz_tail_score > fuzzy_threshold:
                best_match = head if fuzz_head_score > fuzz_tail_score else tail
                best_relation = triplet_type
                break  # Fuzzy match found, no need to continue for this definition

            # Step 3: Semantic similarity (use BERT embeddings)
            definition_embedding = get_bert_embedding(concept_definition)
            head_embedding = get_bert_embedding(head)
            tail_embedding = get_bert_embedding(tail)

            similarity_with_head = calculate_similarity(definition_embedding, head_embedding)
            similarity_with_tail = calculate_similarity(definition_embedding, tail_embedding)

            # Update the best match if semantic similarity exceeds the threshold and is the highest so far
            if similarity_with_head > similarity_threshold and similarity_with_head > max_similarity:
                max_similarity = similarity_with_head
                best_match = head
                best_relation = triplet_type

            if similarity_with_tail > similarity_threshold and similarity_with_tail > max_similarity:
                max_similarity = similarity_with_tail
                best_match = tail
                best_relation = triplet_type

        # If no match is found, use the related terms approach
        if not best_match:
            related_heads = get_related_terms(head, similarity_threshold=0.7)
            related_tails = get_related_terms(tail, similarity_threshold=0.7)
            all_related_terms = related_heads[head] + related_tails[tail]

            for related_term, score in all_related_terms:
                if related_term in concept_definition:
                    best_match = related_term
                    best_relation = triplet_type
                    break

        # If a match is found, update the concept's Type_relation and Concept_of_type_relation
        if best_match:
            concepts_df.at[j, 'Type_relation'] = best_relation
            concepts_df.at[j, 'Concept_of_type_relation'] = best_match

    return concepts_df



In [88]:
updated_concepts_df = update_concept_relations(cleaned_triplet_df,data_final_df, similarity_threshold=0.75, fuzzy_threshold=80)

In [90]:
updated_concepts_df = updated_concepts_df.drop(index=0)

In [91]:
updated_concepts_df = updated_concepts_df.reset_index(drop=True)

In [95]:
htt

,Concepts,Definition,Type_relation,Concept_of_type_relation,Reference,Figure,Synonym,Process_name
0,project risk management,define the project risk management concept. t ...,facet of,project management,[' plan risk management'],,"budget, objective, project, project risk manag...",management recognized projects
1,project risk management,define the project risk management concept. t ...,practiced by,project management practitioners,[' plan risk management'],,"budget, objective, project, project risk manag...",management recognized projects
2,risk management,define the risk management concept. t he defi ...,studies,manage risk,[' plan risk management'],,"identify risks, objective, process, project, p...",management recognized projects
3,project management,define the project management concept. then th...,has part,project management process,[' plan risk management'],,"objective, process, project, project managemen...",management recognized projects
4,project management plan,define the project management plan concept. t ...,use,project management,[' management plan.'],,"budget, execute, process, project, project man...",management recognized projects
...,...,...,...,...,...,...,...,...
62,monte carlo simulation,define the monte carlo simulation concept. the...,uses,probability distribution,[' perform quantitative risk analysis'],,"objective, process, project, project life cycl...",None
63,realism,define the realism concept. t he defi nition o...,facet of,realism,[' introduction'],,"objective, output, process, project, project m...",None
64,critical chain,define the critical chain concept. a ll the cr...,subclass of,project management,[' analysis process'],,"objective, perform qualitative risk analysis, ...",None
65,monitor and control risks,define the monitor and control risks concept. ...,subclass of,manage risk,[' monitor and control risks'],,"monitor, process, project, project risk manage...",stakeholders gain risks


In [96]:
htt = htt[['Concepts', 'Type_relation', 'Concept_of_type_relation','Definition', 'Synonym','Reference',  'Process_name','Figure' ]]

In [97]:
htt

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure
0,project risk management,facet of,project management,define the project risk management concept. t ...,"budget, objective, project, project risk manag...",[' plan risk management'],management recognized projects,
1,project risk management,practiced by,project management practitioners,define the project risk management concept. t ...,"budget, objective, project, project risk manag...",[' plan risk management'],management recognized projects,
2,risk management,studies,manage risk,define the risk management concept. t he defi ...,"identify risks, objective, process, project, p...",[' plan risk management'],management recognized projects,
3,project management,has part,project management process,define the project management concept. then th...,"objective, process, project, project managemen...",[' plan risk management'],management recognized projects,
4,project management plan,use,project management,define the project management plan concept. t ...,"budget, execute, process, project, project man...",[' management plan.'],management recognized projects,
...,...,...,...,...,...,...,...,...
62,monte carlo simulation,uses,probability distribution,define the monte carlo simulation concept. the...,"objective, process, project, project life cycl...",[' perform quantitative risk analysis'],None,
63,realism,facet of,realism,define the realism concept. t he defi nition o...,"objective, output, process, project, project m...",[' introduction'],None,
64,critical chain,subclass of,project management,define the critical chain concept. a ll the cr...,"objective, perform qualitative risk analysis, ...",[' analysis process'],None,
65,monitor and control risks,subclass of,manage risk,define the monitor and control risks concept. ...,"monitor, process, project, project risk manage...",[' monitor and control risks'],stakeholders gain risks,


In [94]:
updated_concepts_df

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure
0,project risk management definition,facet of,effect,define the project risk management de nition c...,"objective, process, project, project managemen...",[' 1.2 project risk management defi nition'],management recognized projects,
1,project risk management in project management ...,facet of,uncertainty,define the role of project risk management in ...,"identify risks, plan risk management, process,...",[' 1.3 role of project risk management in proj...,management recognized projects,
2,good risk management practice,subclass of,defi nition of risk,define the good risk management practice conce...,"objective, process, product, project, risk",[' plan risk management'],management recognized projects,
3,project risk management success factors,has part,pmbok g uide,define the critical success factors for projec...,"project, project risk management, risk, standard",[' 1.5 critical success factors for project ri...,management recognized projects,
4,project risk definition,subclass of,defi nition of project risk,define the de nition of project risk concept. ...,"monitor, project, project management, risk, ri...",[' 2.2 defi nition of project risk'],actions manage risk,
...,...,...,...,...,...,...,...,...
143,risk audits,facet of,uncertainty,define the risk audits concept. t he term risk...,"objective, process, project, risk, risk audits",[' perform quantitative risk analysis'],actions manage risk,
144,risk reassessment,facet of,uncertainty,define the risk reassessment concept. t he pro...,"execute, objective, process, project, risk",[' risk reassessment are:'],actions manage risk,
145,status meetings,subclass of,monitor and control risks,define the status meetings concept. t he proje...,"identify risks, monitor, project, risk, risk m...",[' analysis process'],status reports concept,
146,trend analysis,facet of,uncertainty,define the trend analysis concept. t he term p...,"objective, process, project, risk, standard",[' analysis process'],analysis taken thresholds,


In [98]:
df_combined = pd.concat([htt, updated_concepts_df], ignore_index=True)

In [99]:
df_combined = df_combined.reset_index(drop=True)

In [100]:
df_combined

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure
0,project risk management,facet of,project management,define the project risk management concept. t ...,"budget, objective, project, project risk manag...",[' plan risk management'],management recognized projects,
1,project risk management,practiced by,project management practitioners,define the project risk management concept. t ...,"budget, objective, project, project risk manag...",[' plan risk management'],management recognized projects,
2,risk management,studies,manage risk,define the risk management concept. t he defi ...,"identify risks, objective, process, project, p...",[' plan risk management'],management recognized projects,
3,project management,has part,project management process,define the project management concept. then th...,"objective, process, project, project managemen...",[' plan risk management'],management recognized projects,
4,project management plan,use,project management,define the project management plan concept. t ...,"budget, execute, process, project, project man...",[' management plan.'],management recognized projects,
...,...,...,...,...,...,...,...,...
210,risk audits,facet of,uncertainty,define the risk audits concept. t he term risk...,"objective, process, project, risk, risk audits",[' perform quantitative risk analysis'],actions manage risk,
211,risk reassessment,facet of,uncertainty,define the risk reassessment concept. t he pro...,"execute, objective, process, project, risk",[' risk reassessment are:'],actions manage risk,
212,status meetings,subclass of,monitor and control risks,define the status meetings concept. t he proje...,"identify risks, monitor, project, risk, risk m...",[' analysis process'],status reports concept,
213,trend analysis,facet of,uncertainty,define the trend analysis concept. t he term p...,"objective, process, project, risk, standard",[' analysis process'],analysis taken thresholds,


In [102]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [101]:
# Save the DataFrame to a CSV file
csv_file_path = '/kaggle/working/data_final_pmi.csv'
df_combined.to_csv(csv_file_path, index=False)

# Provide the file path for download
csv_file_path

'/kaggle/working/data_final_pmi.csv'

In [ ]:
# Save the DataFrame to a CSV file
csv_file_path = '/kaggle/working/htt.csv'
htt.to_csv(csv_file_path, index=False)

# Provide the file path for download
csv_file_path

In [ ]:
# from Levenshtein import distance as levenshtein_distance

# # Calculate Levenshtein distance between 'Process_name' and 'Concepts'
# data_final_df['Levenshtein'] = df_final.apply(lambda row: levenshtein_distance(str(row['Concepts']), str(row['Process_name'])), axis=1)

# # Display average Levenshtein distance
# average_levenshtein = data_final_df['Levenshtein'].mean()
# print(f"Average Levenshtein Distance: {average_levenshtein}")


In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # Convert text data into TF-IDF vectors
# vectorizer = TfidfVectorizer()
# tfidf_matrix = vectorizer.fit_transform(data_final_df['Concepts'].astype(str))
# tfidf_matrix_process = vectorizer.transform(data_final_df['Synonym'].astype(str))

# # Calculate cosine similarity
# data_final_df['TFIDF_similarity'] = cosine_similarity(tfidf_matrix, tfidf_matrix_process).diagonal()

# # Display average TF-IDF similarity
# average_tfidf_similarity = data_final_df['TFIDF_similarity'].mean()
# print(f"Average TF-IDF Similarity: {average_tfidf_similarity}")